# RAG - Pipeline with Offline and Online Processing

## Init

In [1]:
import os, sys
sys.path.append(os.path.join(".."))

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from src import VectorStore, FlatIndex, HNSWIndex, SentenceTransformerEmbedding, Retriever, BM25Retriever, RankFusion

C:\Users\PC\AppData\Local\Temp\ipykernel_30196\2752498169.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\code\github\ai\having_fun\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configs

In [2]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
EMBEDDING_MODEL = SentenceTransformerEmbedding
INDEX_TYPE = HNSWIndex
OUTPUT_DIR = "vector_db"

## Pipeline

### Offline Processing

#### Load PDF

In [8]:
def load_document(file_path: str):
    """
    USAGE: documents = load_document(file_path)
    """
    loader = PyPDFLoader(file_path)
    return loader.load()

#### Verify

In [4]:
def verify(results):
    for i, result in enumerate(results):
        print(f"Rank {i+1}")
        print("Score: ", result.score)
        print("Metadata: ", result.document.metadata)
        print(result.document.page_content[:250])
        print("="*10)

#### Full Offline Process

In [5]:
def process_document(pdf_path: str, output_dir=OUTPUT_DIR, embedding_model=EMBEDDING_MODEL, index=INDEX_TYPE, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    documents = load_document(pdf_path)
    store = VectorStore(embedding_model=embedding_model(), index=index())
    store.index_documents(documents, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    store.save(output_dir)
    
    return store

In [7]:
def build_bm25(pdf_path: str):
    documents = load_document(pdf_path)
    bm25 = BM25Retriever()
    bm25.build(documents)
    return bm25

### Online Processing

#### Prompt Generation

In [6]:
def generate_prompt(docs, query: str):
    prompt = '''Given this context: \n'''
    for doc in docs:
        prompt += "#"*10 + "\n"
        prompt += "Title: " + doc.chunk.metadata["title"] + " - Page " + str(doc.chunk.metadata["page"]) + "\n"
        prompt += doc.chunk.text + "\n"
    
    prompt += "#"*10 + "\n"
    prompt += "Question:" + "\n"
    prompt += query
    return prompt


## Test

In [9]:
pdf_path = r"G:\My Drive\0-Studies\algorithm\clrs\Introduction.to.Algorithms.4th.Edition.pdf"

In [ ]:
store = process_document(pdf_path=pdf_path)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2672.52it/s]


In [3]:
# Load vector db
store = VectorStore(embedding_model=EMBEDDING_MODEL, index=INDEX_TYPE).load("vector_db", embedding_model=EMBEDDING_MODEL(), index_class=INDEX_TYPE)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4215.22it/s]


In [10]:
bm25 = build_bm25(pdf_path)

AttributeError: 'Document' object has no attribute 'text'

In [4]:
query = "Dijkstra's algorithm?"

In [10]:
# Search

results = store.search(
    query="Dijkstra's algorithm",
    k=5
)

print(results)

[{'chunk': Chunk(id=2035, text='Just as in Prim’s algorithm, the running time of Dijkstra’s algorithm\ndepends on the speciﬁc implementation of the min-priority queue Q. A\nsimple implementation takes advantage of the vertices being numbered', metadata={'producer': 'calibre (5.37.0) [http://calibre-ebook.com]', 'creator': 'calibre (5.37.0) [http://calibre-ebook.com]', 'creationdate': '2022-04-10T16:55:05+00:00', 'author': 'Thomas H. Cormen;Charles E. Leiserson;Ronald L. Rivest;Clifford Stein; & Charles E. Leiserson & Ronald L. Rivest & Clifford Stein', 'moddate': '2022-04-10T12:55:06-04:00', 'title': 'Introduction to Algorithms, Fourth Edition', 'source': 'G:\\My Drive\\0-Studies\\algorithm\\clrs\\Introduction.to.Algorithms.4th.Edition.pdf', 'total_pages': 1677, 'page': 807, 'page_label': '808'}), 'score': np.float32(0.56688863)}, {'chunk': Chunk(id=2028, text='Figure 22.6 The execution of Dijkstra’s algorithm. The source s is the leftmost vertex. The\nshortest-path estimates appear wi

In [19]:
print(results[0])

{'chunk': Chunk(id=2035, text='Just as in Prim’s algorithm, the running time of Dijkstra’s algorithm\ndepends on the speciﬁc implementation of the min-priority queue Q. A\nsimple implementation takes advantage of the vertices being numbered', metadata={'producer': 'calibre (5.37.0) [http://calibre-ebook.com]', 'creator': 'calibre (5.37.0) [http://calibre-ebook.com]', 'creationdate': '2022-04-10T16:55:05+00:00', 'author': 'Thomas H. Cormen;Charles E. Leiserson;Ronald L. Rivest;Clifford Stein; & Charles E. Leiserson & Ronald L. Rivest & Clifford Stein', 'moddate': '2022-04-10T12:55:06-04:00', 'title': 'Introduction to Algorithms, Fourth Edition', 'source': 'G:\\My Drive\\0-Studies\\algorithm\\clrs\\Introduction.to.Algorithms.4th.Edition.pdf', 'total_pages': 1677, 'page': 807, 'page_label': '808'}), 'score': np.float32(0.56688863)}


In [18]:
prompt = generate_prompt(results, "Dijkstra's algorithm?")
print(prompt)

Given this context: 
##########
Title: Introduction to Algorithms, Fourth Edition - Page 807
Just as in Prim’s algorithm, the running time of Dijkstra’s algorithm
depends on the speciﬁc implementation of the min-priority queue Q. A
simple implementation takes advantage of the vertices being numbered
##########
Title: Introduction to Algorithms, Fourth Edition - Page 805
Figure 22.6 The execution of Dijkstra’s algorithm. The source s is the leftmost vertex. The
shortest-path estimates appear within the vertices, and blue edges indicate predecessor values.
Blue vertices belong to the set S, and tan vertices are in the min-priority queue Q = V − S. (a)
The situation just before the ﬁrst iteration of the while loop of lines 6–12. (b)–(f) The situation
after each successive iteration of the while loop. In each part, the vertex highlighted in orange
was chosen as vertex u in line 7, and each edge highlighted in orange caused a d value and a
predecessor to change when the edge was relaxed. Th

In [5]:
# Using Retriever

retriever = Retriever(store)
results = retriever.retrieve(query, k=5)
print(results)

[SearchResult(chunk=Chunk(id=2028, text='Figure 22.6 The execution of Dijkstra’s algorithm. The source s is the leftmost vertex. The\nshortest-path estimates appear within the vertices, and blue edges indicate predecessor values.\nBlue vertices belong to the set S, and tan vertices are in the min-priority queue Q = V − S. (a)\nThe situation just before the ﬁrst iteration of the while loop of lines 6–12. (b)–(f) The situation\nafter each successive iteration of the while loop. In each part, the vertex highlighted in orange\nwas chosen as vertex u in line 7, and each edge highlighted in orange caused a d value and a\npredecessor to change when the edge was relaxed. The d values and predecessors shown in part\n(f) are the ﬁnal values.\nBecause Dijkstra’s algorithm always chooses the “lightest” or', metadata={'producer': 'calibre (5.37.0) [http://calibre-ebook.com]', 'creator': 'calibre (5.37.0) [http://calibre-ebook.com]', 'creationdate': '2022-04-10T16:55:05+00:00', 'author': 'Thomas H. 

In [8]:
prompt = generate_prompt(results, query)
print(prompt)

Given this context: 
##########
Title: Introduction to Algorithms, Fourth Edition - Page 805
Figure 22.6 The execution of Dijkstra’s algorithm. The source s is the leftmost vertex. The
shortest-path estimates appear within the vertices, and blue edges indicate predecessor values.
Blue vertices belong to the set S, and tan vertices are in the min-priority queue Q = V − S. (a)
The situation just before the ﬁrst iteration of the while loop of lines 6–12. (b)–(f) The situation
after each successive iteration of the while loop. In each part, the vertex highlighted in orange
was chosen as vertex u in line 7, and each edge highlighted in orange caused a d value and a
predecessor to change when the edge was relaxed. The d values and predecessors shown in part
(f) are the ﬁnal values.
Because Dijkstra’s algorithm always chooses the “lightest” or
##########
Title: Introduction to Algorithms, Fourth Edition - Page 835
size of the input, it can be reduced to be linear in the size of the input
usin